# Test Hyrbid Approach

## Setup

In [20]:
!pip3 install -r requirements.txt

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 11.1 MB 4.6 MB/s eta 0:00:01
     |████████████████████████████████| 17.9 MB 23.3 MB/s eta 0:00:01
     |████████████████████████████████| 345 kB 13.8 MB/s eta 0:00:01
     |████████████████████████████████| 1.6 MB 10.0 MB/s eta 0:00:01
     |████████████████████████████████| 30.3 MB 13.3 MB/s eta 0:00:01
     |████████████████████████████████| 2.0 MB 14.2 MB/s eta 0:00:01
     |████████████████████████████████| 444 kB 20.2 MB/s eta 0:00:01
     |████████████████████████████████| 2.7 MB 12.5 MB/s eta 0:00:01
     |████████████████████████████████| 98 kB 16.9 MB/s eta 0:00:01
     |████████████████████████████████| 243 kB 13.5 MB/s eta 0:00:01
     |████████████████████████████████| 67 kB 8.8 MB/s  eta 0:00:01
  distutils: /private/var/folders/6q/t_x9rj512f34ydrk3wdgzq7w0000gn/T/pip-build-env-kf5xmw3w/normal/lib/python3.9/site-packages
  sysconfig: /Library/Python/3.9/si

In [22]:
!pip3 install langchain_huggingface

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 438 kB 6.4 MB/s eta 0:00:01
     |████████████████████████████████| 65 kB 10.7 MB/s eta 0:00:01
     |████████████████████████████████| 363 kB 14.2 MB/s eta 0:00:01
     |████████████████████████████████| 54 kB 9.6 MB/s  eta 0:00:01
     |████████████████████████████████| 633 kB 25.5 MB/s eta 0:00:01
  Attempting uninstall: packaging
    Found existing installation: packaging 25.0
    Uninstalling packaging-25.0:
      Successfully uninstalled packaging-25.0
You should consider upgrading via the '/Applications/Xcode.app/Contents/Developer/usr/bin/python3 -m pip install --upgrade pip' command.


In [60]:
import sys
!{sys.executable} -m pip install nltk

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 22.0 MB/s eta 0:00:00

[notice] A new release of pip is available: 24.2 -> 25.1.1
[notice] To update, run: /opt/homebrew/Cellar/jupyterlab/4.2.5_1/libexec/bin/python -m pip install --upgrade pip


In [61]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /Users/dd/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [64]:
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /Users/dd/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

## Test Hybrid Approach

In [66]:
!python3 test_hybrid.py

Working directory: /Users/dd/Dd/HSLU/4. Semester/2. Advanced GenAI/news-qa-ethz1


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


/Users/dd/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
Initializing Hybrid Retrieval System...

1. Initializing BM25...
Loading HKNews documents...
Loading documents from HKNews...
Skipping document with empty content: HKNews/en_internal/2022/10/should-eth-introduce-rules-for-meetings.json
Skipping document with empty content: HKNews/en_news_events/2022/11/protein-shapes-indicate-parkinsons-disease.json
Skipping document with empty content: HKNews/en_news_events/2022/11/art-created-by-computers.json
Skipping document with empty content: HKNews/en_news_events/2022/10/annette-oxenius-receives-the-cloetta-prize.json
Skipping document with empty content: HKNews/en_news_events/2022/10/trying-on-clothes-virtually.json
Skipping document with empty content: HKNews/en_news_events/2022

## Chroma db conenction failed, fix this

In [34]:
"""
Dense/ChromaDB Adapter for Hybrid Retrieval System
Wraps ChromaDB vector search to provide a unified interface
"""
import sys
import os
#sys.path.append(os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath(__file__)))))

import chromadb
from chromadb.utils import embedding_functions
from typing import List, Dict, Any, Tuple
import json


from pathlib import Path

class DenseAdapter:
    """Adapter for ChromaDB dense retrieval to work with hybrid system"""

    def __init__(self, 
                 chroma_db_path: str = None,
                 collection_name: str = "news_multilingual_recursive",
                 model_name: str = "paraphrase-multilingual-MiniLM-L12-v2"):
        """
        Initialize Dense/ChromaDB adapter.

        Args:
            chroma_db_path: Path to ChromaDB database. If None, auto-detect path.
            collection_name: Name of the collection to use
            model_name: Sentence-transformer model to use
        """

        # 🔁 Auto-detect chroma_db path if not provided
        if chroma_db_path is None:
            # Assume this script runs from `hybrid_retrieval/`, go one level up
            chroma_db_path = Path.cwd().parent / "notebooks" / "chroma_db"

        chroma_db_path = str(chroma_db_path)  # Ensure string type

        # 🔄 Optional reassembly from parts if missing
        if not os.path.exists(chroma_db_path) and os.path.exists("chromadb_parts"):
            print("ChromaDB not found, attempting to reassemble from parts...")
            from scripts.chromadb_utils import reassemble_chromadb
            reassemble_chromadb(".", chroma_db_path)

        # ✅ Initialize ChromaDB client
        self.client = chromadb.PersistentClient(path=chroma_db_path)

        # Embedding model
        self.embedding_function = embedding_functions.SentenceTransformerEmbeddingFunction(
            model_name=model_name
        )

        # Load or create collection
        try:
            self.collection = self.client.get_collection(
                name=collection_name,
                embedding_function=self.embedding_function
            )
        except:
            self.collection = self.client.create_collection(
                name=collection_name,
                embedding_function=self.embedding_function
            )

        self.name = "Dense_ChromaDB"
        print(f"✅ ChromaDB initialized with {self.collection.count():,} documents from '{collection_name}'")
    
    def retrieve(self, query: str, top_k: int = 10) -> List[Tuple[Dict[str, Any], float]]:
        """
        Retrieve documents using dense/vector search
        
        Args:
            query: Search query
            top_k: Number of results to return
            
        Returns:
            List of tuples (document_dict, score)
        """
        # Query the collection
        results = self.collection.query(
            query_texts=[query],
            n_results=top_k,
            include=['documents', 'metadatas', 'distances']
        )
        
        # Standardize the output format
        standardized_results = []
        
        # ChromaDB returns lists of lists, we need the first list
        documents = results['documents'][0] if results['documents'] else []
        metadatas = results['metadatas'][0] if results['metadatas'] else []
        distances = results['distances'][0] if results['distances'] else []
        
        for idx, (doc, metadata, distance) in enumerate(zip(documents, metadatas, distances)):
            # Convert distance to similarity score (inverse of distance)
            score = 1.0 / (1.0 + distance)
            
            doc_dict = {
                'id': metadata.get('id', f"dense_{idx}"),
                'content': doc,
                'title': metadata.get('title', ''),
                'metadata': metadata,
                'retriever': self.name,
                'distance': distance
            }
            standardized_results.append((doc_dict, score))
        
        return standardized_results
    
    def get_name(self) -> str:
        """Return the name of this retriever"""
        return self.name

In [37]:
retriever = DenseAdapter()
results = retriever.retrieve("When did the InSight get to Mars?", top_k=16)

for doc, score in results:
    print(f"🔎 Score: {score:.3f} | Title: {doc['title']}")
    print(f"📄 Content: {doc['content'][:100]}...\n")

✅ ChromaDB initialized with 109,128 documents from 'news_multilingual_recursive'
🔎 Score: 0.085 | Title: 25 Jahre Entscheidungshilfe Fuer Die Schulleitung
📄 Content: Cornell University in den USA hinter sich. (Fast) alles sprach für eine Karriere in der Wissenschaft...

🔎 Score: 0.084 | Title: 25 Jahre Entscheidungshilfe Fuer Die Schulleitung
📄 Content: Altersrücktritte...

🔎 Score: 0.082 | Title: 25 Jahre Entscheidungshilfe Fuer Die Schulleitung
📄 Content: auch eigene Ansichten zu hinterfragen. Sonst bleibt man stehen.»...

🔎 Score: 0.082 | Title: A Good Solutions Secret
📄 Content: the accuracy of a simulation comes from Geophysics research. Together with Yunan Yang, who most rece...

🔎 Score: 0.081 | Title: 25 Jahre Entscheidungshilfe Fuer Die Schulleitung
📄 Content: Doch es kam anders. Mit der Neubesetzung von Professuren verschoben sich die Forschungsschwerpunkte....

🔎 Score: 0.079 | Title: 25 Jahre Entscheidungshilfe Fuer Die Schulleitung
📄 Content: Dienstjubiläen April 2014...



In [30]:
import os
print("Current working dir:", os.getcwd())

Current working dir: /Users/dd/Dd/HSLU/4. Semester/2. Advanced GenAI/news-qa-ethz1/hybrid_retrieval


In [43]:
import chromadb
from pathlib import Path

chroma_path = Path("/Users/dd/Dd/HSLU/4. Semester/2. Advanced GenAI/notebooks/chroma_db")
client = chromadb.PersistentClient(path=str(chroma_path))

collections = client.list_collections()
print(f"\n📦 Collections in: {chroma_path}")
for col in collections:
    try:
        count = col.count()
        print(f"✅ {col.name}: {count:,} documents")
    except Exception as e:
        print(f"⚠️ {col.name}: error -> {e}")


📦 Collections in: /Users/dd/Dd/HSLU/4. Semester/2. Advanced GenAI/notebooks/chroma_db
✅ ethz_news: 47,333 documents
✅ news_multilingual_recursive: 0 documents


In [48]:
retriever = DenseAdapter(
    chroma_db_path="/Users/dd/Dd/HSLU/4. Semester/2. Advanced GenAI/news-qa-ethz1/notebooks/chroma_db",
    collection_name="news_multilingual_recursive"
)

✅ ChromaDB initialized with 109,128 documents from 'news_multilingual_recursive'


# Evaluate on Benchmark Questions

In [69]:
!python3 evaluate_reranking_benchmark.py


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


/Users/dd/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
RERANKING EVALUATION ON 25 BENCHMARK QUESTIONS
Loaded 25 benchmark questions

Initializing retrievers...
Loading HKNews documents...
Loading documents from HKNews...
Skipping document with empty content: HKNews/en_internal/2022/10/should-eth-introduce-rules-for-meetings.json
Skipping document with empty content: HKNews/en_news_events/2022/11/protein-shapes-indicate-parkinsons-disease.json
Skipping document with empty content: HKNews/en_news_events/2022/11/art-created-by-computers.json
Skipping document with empty content: HKNews/en_news_events/2022/10/annette-oxenius-receives-the-cloetta-prize.json
Skipping document with empty content: HKNews/en_news_events/2022/10/trying-on-clothes-virtually.json
Skipping document with e

## New evaluation scipt for reranking quality

evaluate based on ground truth answers (rather than document IDs)

	1.	Loads the benchmark_qa.json file
	2.	Uses a sentence transformer to compute semantic similarity
	3.	Computes average relevance across top-5 documents
	4.	Outputs precision-style and similarity metrics for baseline vs reranked results

In [75]:
!python3 evaluate_semantic_similarity.py

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


/Users/dd/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
Loading HKNews documents...
Loading documents from HKNews...
Error: Directory HKNews not found
📂 Loading ChromaDB from: /Users/dd/Dd/HSLU/4. Semester/2. Advanced GenAI/news-qa-ethz1/notebooks/chroma_db
✅ ChromaDB initialized with 109,128 documents from 'news_multilingual_recursive'
Loading knowledge graph from graph.pickle...
Error loading graph: [Errno 2] No such file or directory: 'graph.pickle'
Loading entity chunks from entity_chunks.pickle...
Error loading entity chunks: [Errno 2] No such file or directory: 'entity_chunks.pickle'
Generating article embeddings...
Generated embeddings for 0 articles
Initializing Multi-Model Reranker System...
TinyBERT model loaded (fast)
MiniLM model loaded (balanced)
BGE reranker load

In [78]:
!python3 evaluate_semantic_similarity.py

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


/Users/dd/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
SEMANTIC SIMILARITY EVALUATION BASED ON GROUND TRUTH ANSWERS
Loaded 25 benchmark questions
Loading HKNews documents...
Loading documents from HKNews...
Skipping document with empty content: HKNews/en_internal/2022/10/should-eth-introduce-rules-for-meetings.json
Skipping document with empty content: HKNews/en_news_events/2022/11/protein-shapes-indicate-parkinsons-disease.json
Skipping document with empty content: HKNews/en_news_events/2022/11/art-created-by-computers.json
Skipping document with empty content: HKNews/en_news_events/2022/10/annette-oxenius-receives-the-cloetta-prize.json
Skipping document with empty content: HKNews/en_news_events/2022/10/trying-on-clothes-virtually.json
Skipping document with empty content: 

# Build a RAG pipeline

In [80]:
import sys
sys.path.append('/Users/dd/Dd/HSLU/4. Semester/2. Advanced GenAI/news-qa-ethz1/')

In [85]:
!pip3 install deep-translator rank_bm25

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Applications/Xcode.app/Contents/Developer/usr/bin/python3 -m pip install --upgrade pip' command.


In [89]:
! /opt/homebrew/Cellar/jupyterlab/4.2.5_1/libexec/bin/python -m pip install deep-translator rank_bm25

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



[notice] A new release of pip is available: 24.2 -> 25.1.1
[notice] To update, run: /opt/homebrew/Cellar/jupyterlab/4.2.5_1/libexec/bin/python -m pip install --upgrade pip


In [101]:
import os
from typing import List, Dict
from transformers import pipeline
from sentence_transformers import SentenceTransformer, util
from chromadb import PersistentClient

from pre_retrieval import PreRetrievalPipeline, QueryRouter
from hybrid_retrieval.hybrid_retriever import HybridRetriever
from hybrid_retrieval.adapters.bm25_adapter import BM25Adapter
from hybrid_retrieval.adapters.dense_adapter import DenseAdapter
from hybrid_retrieval.adapters.graphrag_adapter import GraphRAGAdapter


class RAGPipeline:
    def __init__(
        self,
        collection_name: str = "news_multilingual_recursive",
        embedding_model: str = "paraphrase-MiniLM-L6-v2",
        gen_model: str = "google/flan-t5-base",
        chroma_path: str = "./chroma_db"
    ):
        # Initialize components
        self.chroma_client = PersistentClient(path=chroma_path)
        self.collection_name = collection_name

        # Pre-retrieval
        self.preprocessor = PreRetrievalPipeline(self.chroma_client, collection_name)

        # Adapters
        self.bm25 = BM25Adapter()
        self.dense = DenseAdapter()
        self.graphrag = GraphRAGAdapter()

        # Hybrid retriever (can rerank or not)
        self.retriever = HybridRetriever(
            bm25_adapter=self.bm25,
            dense_adapter=self.dense,
            graphrag_adapter=self.graphrag,
            use_reranking=True
        )

        # Reranker model
        self.reranker = SentenceTransformer(embedding_model)

        # Generator
        self.generator = pipeline("text2text-generation", model=gen_model)

    def retrieve_documents(self, query: str, top_k: int = 10) -> List[Dict]:
        # Preprocess query
        context = self.preprocessor.process_query(query)
        rewritten = context.processed_query

        # Routing (if needed)
        routing_config = QueryRouter.route_query(context)

        # Retrieve top documents (reranking is inside retriever)
        retrieved = self.retriever.retrieve(
            rewritten,
            top_k=top_k,
            rerank_method="ensemble"
        )

        return [doc for doc, _ in retrieved]

    def generate_answer(self, query: str, context_docs: List[Dict], max_length: int = 256) -> str:
        if not context_docs:
            return "No relevant documents found to answer this question."

        context_text = " ".join([doc.get("text", "") for doc in context_docs[:3]])
        prompt = f"Question: {query}\nContext: {context_text}\nAnswer:"

        result = self.generator(prompt, max_length=max_length, do_sample=False, truncation=True)
        return result[0]["generated_text"].strip()

    def run(self, query: str, top_k: int = 10) -> Dict:
        print(f"\n🔍 QUERY: {query}")
        documents = self.retrieve_documents(query, top_k=top_k)
        answer = self.generate_answer(query, documents)

        return {
            "query": query,
            "answer": answer,
            "documents_used": [doc["id"] for doc in documents]
        }


# Usage
if __name__ == "__main__":
    rag = RAGPipeline()
    
    test_query = "Who at ETH received ERC grants?"
    result = rag.run(test_query)

    print("\n📢 GENERATED ANSWER:")
    print(result["answer"])
    print("\n📄 DOCS USED:")
    print(result["documents_used"])

Loading HKNews documents...
Loading documents from HKNews...
Error: Directory HKNews not found
📂 Loading ChromaDB from: /Users/dd/Dd/HSLU/4. Semester/2. Advanced GenAI/news-qa-ethz1/notebooks/chroma_db
✅ ChromaDB initialized with 109,128 documents from 'news_multilingual_recursive'
Loading knowledge graph from graph.pickle...
Error loading graph: [Errno 2] No such file or directory: 'graph.pickle'
Loading entity chunks from entity_chunks.pickle...
Error loading entity chunks: [Errno 2] No such file or directory: 'entity_chunks.pickle'
Generating article embeddings...
Generated embeddings for 0 articles
Initializing Multi-Model Reranker System...
TinyBERT model loaded (fast)
MiniLM model loaded (balanced)
BGE reranker loaded (sota)
Loaded 3 cross-encoder models
Multi-model reranker initialized successfully
Available methods: ['tinybert', 'minilm', 'bge', 'ensemble', 'summary_fusion', 'keyword_boost', 'cohere']


Device set to use mps:0



🔍 QUERY: Who at ETH received ERC grants?

HYBRID RETRIEVAL WITH RERANKING
Query: Who at ETH Zurich Swiss Federal Institute Technology received European Research Council ERC grants? universität stipendium university Who
BM25 retriever not initialized
BM25: Retrieved 0 results
Dense_ChromaDB: Retrieved 20 results
Processing query: Who at ETH Zurich Swiss Federal Institute Technology received European Research Council ERC grants? universität stipendium university Who
Detected language: de
Local retrieval found 0 results
Global expansion found 0 results
Returning 0 final results
GraphRAG: Retrieved 0 results
After fusion: 20 results
Applying reranking using method: ensemble
Reranking 20 documents using method: ensemble


Both `max_new_tokens` (=256) and `max_length`(=256) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Reranking applied with ensemble. Final results: 10

📢 GENERATED ANSWER:
the University of Cambridge

📄 DOCS USED:
['dense_0', 'dense_2', 'dense_16', 'dense_5', 'dense_3', 'dense_9', 'dense_11', 'dense_13', 'dense_6', 'dense_17']


In [99]:
! /opt/homebrew/Cellar/jupyterlab/4.2.5_1/libexec/bin/python -m pip install hf_xet

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 18.7 MB/s eta 0:00:00

[notice] A new release of pip is available: 24.2 -> 25.1.1
[notice] To update, run: /opt/homebrew/Cellar/jupyterlab/4.2.5_1/libexec/bin/python -m pip install --upgrade pip


In [102]:
# Usage
if __name__ == "__main__":
    rag = RAGPipeline()
    
    test_query = "When did the InSight get to Mars?"
    result = rag.run(test_query)

    print("\n📢 GENERATED ANSWER:")
    print(result["answer"])
    print("\n📄 DOCS USED:")
    print(result["documents_used"])

Loading HKNews documents...
Loading documents from HKNews...
Error: Directory HKNews not found
📂 Loading ChromaDB from: /Users/dd/Dd/HSLU/4. Semester/2. Advanced GenAI/news-qa-ethz1/notebooks/chroma_db
✅ ChromaDB initialized with 109,128 documents from 'news_multilingual_recursive'
Loading knowledge graph from graph.pickle...
Error loading graph: [Errno 2] No such file or directory: 'graph.pickle'
Loading entity chunks from entity_chunks.pickle...
Error loading entity chunks: [Errno 2] No such file or directory: 'entity_chunks.pickle'
Generating article embeddings...
Generated embeddings for 0 articles
Initializing Multi-Model Reranker System...
TinyBERT model loaded (fast)
MiniLM model loaded (balanced)
BGE reranker loaded (sota)
Loaded 3 cross-encoder models
Multi-model reranker initialized successfully
Available methods: ['tinybert', 'minilm', 'bge', 'ensemble', 'summary_fusion', 'keyword_boost', 'cohere']


Device set to use mps:0



🔍 QUERY: When did the InSight get to Mars?

HYBRID RETRIEVAL WITH RERANKING
Query: the InSight get to Mars? When Mars
BM25 retriever not initialized
BM25: Retrieved 0 results
Dense_ChromaDB: Retrieved 20 results
Processing query: the InSight get to Mars? When Mars
Detected language: en
Local retrieval found 0 results
Global expansion found 0 results
Returning 0 final results
GraphRAG: Retrieved 0 results
After fusion: 20 results
Applying reranking using method: ensemble
Reranking 20 documents using method: ensemble
Reranking applied with ensemble. Final results: 10


Both `max_new_tokens` (=256) and `max_length`(=256) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



📢 GENERATED ANSWER:
July 15, 2015

📄 DOCS USED:
['dense_19', 'dense_1', 'dense_13', 'dense_11', 'dense_5', 'dense_18', 'dense_10', 'dense_14', 'dense_0', 'dense_9']


In [10]:
!python3 rag_pipeline.py

/Users/dd/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
Working directory: /Users/dd/Dd/HSLU/4. Semester/2. Advanced GenAI/news-qa-ethz1
Initializing Hybrid Retrieval System...

1. Initializing BM25...
Loading HKNews documents...
Loading documents from HKNews...
Skipping document with empty content: HKNews/en_internal/2022/10/should-eth-introduce-rules-for-meetings.json
Skipping document with empty content: HKNews/en_news_events/2022/11/protein-shapes-indicate-parkinsons-disease.json
Skipping document with empty content: HKNews/en_news_events/2022/11/art-created-by-computers.json
Skipping document with empty content: HKNews/en_news_events/2022/10/annette-oxenius-receives-the-cloetta-prize.json
Skipping document with empty content: HKNews/en_news_events/2022/10/trying-on-clothe

In [111]:
from typing import List, Tuple, Dict

In [8]:
! /opt/homebrew/Cellar/jupyterlab/4.2.5_1/libexec/bin/python -m pip 

ERROR: unknown command "pipeline"


In [12]:
!python3 rag_pipeline.py

/Users/dd/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
Working directory: /Users/dd/Dd/HSLU/4. Semester/2. Advanced GenAI/news-qa-ethz1
Initializing Hybrid Retrieval System...

1. Initializing BM25...
Loading HKNews documents...
Loading documents from HKNews...
Skipping document with empty content: HKNews/en_internal/2022/10/should-eth-introduce-rules-for-meetings.json
Skipping document with empty content: HKNews/en_news_events/2022/11/protein-shapes-indicate-parkinsons-disease.json
Skipping document with empty content: HKNews/en_news_events/2022/11/art-created-by-computers.json
Skipping document with empty content: HKNews/en_news_events/2022/10/annette-oxenius-receives-the-cloetta-prize.json
Skipping document with empty content: HKNews/en_news_events/2022/10/trying-on-clothe